In [9]:
# from googleapiclient.discovery import build

# API_KEY = "API"
# youtube = build("youtube", "v3", developerKey=API_KEY)

# query = "#"
# # start_date = "2023-07-01T00:00:00Z"
# # end_date   = "2023-07-02T00:00:00Z"

# request = youtube.search().list(
#     q=query,
#     part="id",
#     type="video",
#     # publishedAfter=start_date,
#     # publishedBefore=end_date,
#     maxResults=50
# )

# response = request.execute()

# video_ids = [
#     item["id"]["videoId"]
#     for item in response.get("items", [])
#     if "videoId" in item["id"]
# ]

# print(video_ids)


## Search with random seeded words

In [ ]:
import re
from collections import Counter
from googleapiclient.discovery import build

API_KEY = "API"
youtube = build("youtube", "v3", developerKey=API_KEY)

HASHTAG_RE = re.compile(r'(?<!\w)#([0-9A-Za-z_]{2,})')  # require >=2 chars after '#'

def search_video_ids(seed_query, max_pages=5, **kwargs):
    """Collect videoIds from search.list, paginating."""
    video_ids = []
    page_token = None

    for _ in range(max_pages):
        req = youtube.search().list(
            part="id",
            type="video",
            q=seed_query,
            maxResults=50,
            pageToken=page_token,
            **kwargs,
        )
        resp = req.execute()

        for item in resp.get("items", []):
            vid = item.get("id", {}).get("videoId")
            if vid:
                video_ids.append(vid)

        page_token = resp.get("nextPageToken")
        if not page_token:
            break

    return video_ids

def fetch_descriptions(video_ids):
    """Fetch snippet.description for a list of video IDs (batched by 50)."""
    descriptions = []
    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i+50]
        req = youtube.videos().list(
            part="snippet",
            id=",".join(batch),
            fields="items(id,snippet(description))"  # reduce payload/quota usage
        )
        resp = req.execute()
        for item in resp.get("items", []):
            desc = item.get("snippet", {}).get("description", "") or ""
            descriptions.append(desc)
    return descriptions

def extract_hashtags(text):
    return [m.group(1).lower() for m in HASHTAG_RE.finditer(text)]

In [ ]:
seeds = ["the", "and", "to"]

all_hashtags = Counter()

for seed in seeds:
    vids = search_video_ids(
        seed_query=seed,
        max_pages=3,
        order="date",               
        # regionCode="US",
        safeSearch="none",
        # publishedAfter="2026-01-01T00:00:00Z", # optional time
    )

    descs = fetch_descriptions(vids)
    for d in descs:
        all_hashtags.update(extract_hashtags(d))

print(all_hashtags.most_common(50))